This notebook provides a descriptive overview of the unified CGM data, combining subject-level and dataset-level visualizations with summary statistics to characterize glucose behavior over time. We explore glucose trajectories, frequency distributions, and mean profiles (overall and daily), and we report standard CGM metrics (e.g., time-in-range style metrics, CV, and GMI) to ensure clinically meaningful and comparable summaries across datasets.

In [ ]:
import sys
import os

parent_dir = os.path.abspath("../")
sys.path.append(parent_dir)

import polars as pl
from chronoindex.dataset_loader import load_datasets
from chronoindex.dataset_unifier import UnifiedCGMDataset, UnifiedClinicalData

In [ ]:
datasets, errors = load_datasets(
    config_path= "datasets.toml",
    only=["praes_unil"],
)
if errors:
    raise RuntimeError(errors)

In [ ]:
unified = UnifiedCGMDataset(
    datasets
    )

df = unified.unified_data   # same as unified.cgm_data
df.head()

Computing Catch 22 features from daily time series

In [ ]:
from chronoindex.statistical_summaries.catchfeatures import compute_catch22_features


catch = compute_catch22_features(df)
print(catch.select(["dataset", "Id", "date", "catch22_features"]).head())

CGM-derived metrics (adapted from the [iglu](https://github.com/irinagain/iglu/) framework)

In [ ]:
from chronoindex.statistical_summaries.iglu_metrics import (
    summary,
    in_range_percent,
    above_percent,
    below_percent,
    cv_glu,
    gmi,
)

# 1) Summary stats
summary_df = summary(df)
summary_df.head()

In [ ]:
# 2) In-range percentages (default 70-180 and 63-140)
tir_df = in_range_percent(df)
tir_df.head()

In [ ]:
# custom ranges
tir_custom_df = in_range_percent(df, target_ranges=[(70, 180), (63, 140), (80, 140)])

# 3) Above percentages (default 140, 180, 250)
above_df = above_percent(df)
above_df.head()

In [ ]:
# custom thresholds
above_custom_df = above_percent(df, targets_above=[100, 150, 180])

In [ ]:

# 4) Below percentages (default 54, 70)
below_df = below_percent(df)
below_df.head()

In [ ]:
# custom thresholds
below_custom_df = below_percent(df, targets_below=[50, 100, 180])



In [ ]:
# 5) CV
cv_df = cv_glu(df)
cv_df.head()

In [ ]:
# 6) GMI
gmi_df = gmi(df)
gmi_df.head()

Visualization

In [ ]:
import polars as pl
from chronoindex.statistical_summaries.plot import (
    plot_subject_glucose_time_series,
    plot_subject_glucose_frequency,
    plot_subject_mean_glucose,
    plot_dataset_glucose_time_series,
    plot_dataset_glucose_frequency,
    plot_dataset_mean_glucose,
)


# Subject-level plots
plot_subject_glucose_time_series(df, subject_id="s24417854",daily=True) # Can be done on all subjects by removing subject_id parameter.

In [ ]:
plot_subject_glucose_frequency(df,subject_id="s24417854")                          # all subjects 

In [ ]:
plot_subject_mean_glucose(df, subject_id=["s24417854"])  # selected subjects

In [ ]:
plot_dataset_glucose_frequency(df, dataset="praes_unil") 

In [ ]:
plot_dataset_mean_glucose(df) 